In [ ]:
import os

BREAKHIS_PATH = "/kaggle/input/datasets/um/cancer-cnn-classification/BreaKHis_v1"
NCT_PATH = "/kaggle/input/datasets/um/cancer-cnn-classification/NCT-CRC-HE-100K"
ISIC_PATH = "/kaggle/input/datasets/salviohexia/isic-2019-skin-lesion-images-for-classification/ISIC_2019_Training_GroundTruth.csv"

print("BreakHis exists:", os.path.exists(BREAKHIS_PATH))
print("ISIC exists:", os.path.exists(ISIC_PATH))
print("NCT exists:", os.path.exists(NCT_PATH))

In [ ]:
pip install -q imbalanced-learn tabulate

In [ ]:
import json
import random
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.metrics import (accuracy_score, auc, confusion_matrix, f1_score,
                              precision_score, recall_score, roc_auc_score,
                              roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet, DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.mobilenet import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.callbacks import (CSVLogger, EarlyStopping, ModelCheckpoint,
                                         ReduceLROnPlateau)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
})

In [ ]:
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

# NOTE the doubled "BreaKHis_v1/BreaKHis_v1" and "NCT-CRC-HE-100K/NCT-CRC-HE-100K"
# segments below - that's how these two got nested when uploaded, not a typo.
BREAKHIS_DIR = (KAGGLE_INPUT_ROOT / "datasets/um/cancer-cnn-classification"
                 / "BreaKHis_v1" / "BreaKHis_v1" / "histology_slides" / "breast")
NCT_CRC_DIR = (KAGGLE_INPUT_ROOT / "datasets/um/cancer-cnn-classification"
               / "NCT-CRC-HE-100K" / "NCT-CRC-HE-100K")
# This ISIC 2019 mirror ships pre-sorted into one subfolder per class
# (MEL/NV/BCC/AK/BKL/DF/VASC/SCC), unlike the official flat-folder-plus-CSV
# release - the loader in CELL 4 is written for that folder structure.
ISIC_DIR = KAGGLE_INPUT_ROOT / "datasets/salviohexia/isic-2019-skin-lesion-images-for-classification"

OUTPUT_ROOT = Path("/kaggle/working")
PROCESSED_ROOT = OUTPUT_ROOT / "processed"
MODELS_DIR = OUTPUT_ROOT / "outputs" / "models"
LOGS_DIR = OUTPUT_ROOT / "outputs" / "logs"
FIGURES_METHODOLOGY_DIR = OUTPUT_ROOT / "outputs" / "figures" / "methodology"
FIGURES_RESULTS_DIR = OUTPUT_ROOT / "outputs" / "figures" / "results"
TABLES_METHODOLOGY_DIR = OUTPUT_ROOT / "outputs" / "tables" / "methodology"
TABLES_RESULTS_DIR = OUTPUT_ROOT / "outputs" / "tables" / "results"

for d in (PROCESSED_ROOT, MODELS_DIR, LOGS_DIR, FIGURES_METHODOLOGY_DIR,
          FIGURES_RESULTS_DIR, TABLES_METHODOLOGY_DIR, TABLES_RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

ARCHITECTURES = ["vgg16", "resnet50", "mobilenet", "densenet121"]
DATASETS = ["BreakHis", "NCT-CRC-HE-100K", "ISIC_2019"]

TARGET_SIZE = {"vgg16": (224, 224), "resnet50": (224, 224),
                "mobilenet": (224, 224), "densenet121": (224, 224)}
BATCH_SIZE = 32

# Two-phase fine-tuning protocol (phase 1: head only: phase 2: unfreeze top 30%)
FREEZE_BASE_EPOCHS = 5
FINE_TUNE_EPOCHS = 15
FINE_TUNE_UNFREEZE_FRACTION = 0.30
BASE_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
DROPOUT_RATE = 0.30
DENSE_UNITS = 256
EARLY_STOPPING_PATIENCE = 5
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.5

# BreakHis label granularity: "binary" (benign/malignant) or "subtype" (8-way)
BREAKHIS_LABEL_MODE = "binary"

IMBALANCE_STRATEGY = "class_weight"   # "class_weight" | "smote" | "none"
SMOTE_MAX_SAMPLES_PER_CLASS = 1500
SMOTE_THUMBNAIL_SIZE = (32, 32)

GRADCAM_LAST_CONV_LAYER = {
    "vgg16": "block5_conv3",
    "resnet50": "conv5_block3_out",
    "mobilenet": "conv_pw_13_relu",
    "densenet121": "relu",
}
GRADCAM_NUM_EXAMPLES = 6


In [ ]:
VALID_MAGNIFICATIONS = {"40X", "100X", "200X", "400X"}


def load_breakhis_index(root_dir: Path, label_mode: str = "binary") -> pd.DataFrame:
    """BreakHis: benign|malignant/SOB/<tumour_type>/<patient_id>/<magnification>/*.png"""
    root_dir = Path(root_dir)
    if not root_dir.exists():
        raise FileNotFoundError(f"BreakHis root not found at '{root_dir}'.")

    records = []
    for png_path in root_dir.rglob("*.png"):
        parts = png_path.parts
        class_dir = magnification = patient_id = tumour_type = None
        for i, part in enumerate(parts):
            if part in ("benign", "malignant"):
                class_dir = part
            if part in VALID_MAGNIFICATIONS:
                magnification = part
                patient_id = parts[i - 1]
                tumour_type = parts[i - 2]
        label = class_dir if label_mode == "binary" else tumour_type
        records.append({
            "filepath": str(png_path),
            "label": label or "unknown",
            "magnification": magnification or "unknown",
            "patient_id": patient_id or "unknown",
        })

    if not records:
        raise RuntimeError(f"No .png files found under '{root_dir}'.")
    return pd.DataFrame(records)


def load_folder_labelled_index(
    root_dir: Path,
    expected_classes: Optional[set] = None,
    extensions: Sequence[str] = ("*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg"),
) -> pd.DataFrame:
    """
    Generic loader for datasets already sorted into one subfolder per class.
    Used for both NCT-CRC-HE-100K (ADI/BACK/DEB/LYM/MUC/MUS/NORM/STR/TUM) and
    this Kaggle mirror of ISIC 2019 (MEL/NV/BCC/AK/BKL/DF/VASC/SCC).
    """
    root_dir = Path(root_dir)
    if not root_dir.exists():
        raise FileNotFoundError(f"Dataset root not found at '{root_dir}'.")

    class_dirs = [d for d in root_dir.iterdir() if d.is_dir()]
    found = {d.name for d in class_dirs}
    if expected_classes:
        missing = expected_classes - found
        if missing:
            print(f"[warning] Expected class folders not found under '{root_dir}': {sorted(missing)}")

    records = []
    for class_dir in class_dirs:
        for ext in extensions:
            for img_path in class_dir.glob(ext):
                records.append({"filepath": str(img_path), "label": class_dir.name})

    if not records:
        raise RuntimeError(f"No images found under '{root_dir}'.")
    return pd.DataFrame(records)

In [ ]:
def stratified_split(df: pd.DataFrame, train_ratio: float, val_ratio: float,
                      test_ratio: float, label_col: str = "label",
                      random_state: int = 42) -> pd.DataFrame:
    assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-6

    train_df, temp_df = train_test_split(
        df, train_size=train_ratio, stratify=df[label_col], random_state=random_state)
    relative_val = val_ratio / (val_ratio + test_ratio)
    val_df, test_df = train_test_split(
        temp_df, train_size=relative_val, stratify=temp_df[label_col], random_state=random_state)

    train_df = train_df.copy(); train_df["split"] = "train"
    val_df = val_df.copy();     val_df["split"] = "val"
    test_df = test_df.copy();   test_df["split"] = "test"
    return pd.concat([train_df, val_df, test_df], ignore_index=True)


def compute_class_weights(df: pd.DataFrame, label_col: str = "label") -> Dict[int, float]:
    classes = sorted(df[label_col].unique())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    y = df[label_col].map(class_to_idx).values
    weights = compute_class_weight(class_weight="balanced", classes=np.arange(len(classes)), y=y)
    return {class_to_idx[c]: float(w) for c, w in zip(classes, weights)}


def smote_oversample_index(df: pd.DataFrame, label_col: str = "label",
                            thumbnail_size=(32, 32), max_samples_per_class: int = 1500,
                            random_state: int = 42) -> pd.DataFrame:
    """SMOTE on flattened thumbnails - only tractable for the smaller BreakHis split."""
    from imblearn.over_sampling import SMOTE

    thumbnails, labels_list = [], []
    for _, row in df.iterrows():
        try:
            img = Image.open(row["filepath"]).convert("RGB").resize(thumbnail_size)
        except Exception as exc:
            print(f"[warning] Skipping unreadable image '{row['filepath']}': {exc}")
            continue
        thumbnails.append(np.asarray(img).flatten())
        labels_list.append(row[label_col])

    X = np.stack(thumbnails)
    y = np.array(labels_list)
    class_counts = pd.Series(y).value_counts()
    sampling_strategy = {cls: min(max_samples_per_class, int(class_counts.max()))
                          for cls in class_counts.index}

    smote = SMOTE(random_state=random_state, sampling_strategy=sampling_strategy)
    X_resampled, y_resampled = smote.fit_resample(X, y)

    n_original = len(X)
    n_synthetic = len(X_resampled) - n_original
    print(f"[info] SMOTE generated {n_synthetic} synthetic samples ({n_original} -> {len(X_resampled)}).")

    synthetic_records = []
    if n_synthetic > 0:
        synth_dir = Path(df["filepath"].iloc[0]).parent.parent / "_smote_synthetic"
        synth_dir = OUTPUT_ROOT / "smote_synthetic"
        synth_dir.mkdir(parents=True, exist_ok=True)
        for i in range(n_original, len(X_resampled)):
            pixels = X_resampled[i].reshape(*thumbnail_size, 3).astype(np.uint8)
            out_path = synth_dir / f"synthetic_{i}.png"
            Image.fromarray(pixels).save(out_path)
            synthetic_records.append({"filepath": str(out_path), "label": y_resampled[i], "split": "train"})

    synthetic_df = pd.DataFrame(synthetic_records)
    return pd.concat([df, synthetic_df], ignore_index=True) if len(synthetic_df) else df

In [ ]:
def preprocess_dataset(name: str, df: pd.DataFrame, imbalance_strategy: str = IMBALANCE_STRATEGY) -> pd.DataFrame:
    df = stratified_split(df, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, random_state=RANDOM_SEED)

    print(f"\n[{name}] split sizes:")
    print(df["split"].value_counts().to_string())
    print(f"[{name}] class distribution (train split):")
    print(df.loc[df["split"] == "train", "label"].value_counts().to_string())

    train_df = df[df["split"] == "train"]
    class_weights = None

    if imbalance_strategy == "smote":
        train_df = smote_oversample_index(train_df, thumbnail_size=SMOTE_THUMBNAIL_SIZE,
                                           max_samples_per_class=SMOTE_MAX_SAMPLES_PER_CLASS,
                                           random_state=RANDOM_SEED)
        df = pd.concat([train_df, df[df["split"] != "train"]], ignore_index=True)
    elif imbalance_strategy == "class_weight":
        class_weights = compute_class_weights(train_df)

    out_dir = PROCESSED_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / "index.csv", index=False)
    if class_weights is not None:
        with open(out_dir / "class_weights.json", "w") as f:
            json.dump(class_weights, f, indent=2)

    print(f"[{name}] preprocessing complete -> '{out_dir}'")
    return df


breakhis_df = load_breakhis_index(BREAKHIS_DIR, label_mode=BREAKHIS_LABEL_MODE)
preprocess_dataset("BreakHis", breakhis_df)

isic_df = load_folder_labelled_index(
    ISIC_DIR, expected_classes={"MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"})
preprocess_dataset("ISIC_2019", isic_df)

In [ ]:
def stratified_subsample(df, n_per_class, label_col="label", random_state=42):
    class_counts_before = df[label_col].value_counts()
    sampled_parts = []
    for cls, group in df.groupby(label_col):
        n = min(len(group), n_per_class)
        sampled_parts.append(group.sample(n=n, random_state=random_state))
    sampled = pd.concat(sampled_parts, ignore_index=True)
    class_counts_after = sampled[label_col].value_counts()

    print(f"[subsample] {len(df):,} -> {len(sampled):,} images "
          f"({n_per_class} per class, capped by availability)")
    print("[subsample] Per-class counts before -> after:")
    for cls in class_counts_before.index:
        before = class_counts_before.get(cls, 0)
        after = class_counts_after.get(cls, 0)
        print(f"    {cls}: {before} -> {after}")

    return sampled



NCT_CRC_IMAGES_PER_CLASS = 2000 
nct_crc_df = load_folder_labelled_index(
    NCT_CRC_DIR, expected_classes={"ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"})
nct_crc_df = stratified_subsample(nct_crc_df, n_per_class=NCT_CRC_IMAGES_PER_CLASS)
preprocess_dataset("NCT-CRC-HE-100K", nct_crc_df)

In [ ]:
PREPROCESS_FUNCTIONS = {
    "vgg16": vgg16_preprocess,
    "resnet50": resnet50_preprocess,
    "mobilenet": mobilenet_preprocess,
    "densenet121": densenet_preprocess,
}

_augmentation_pipeline = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation_pipeline")


def augment_image(image: tf.Tensor) -> tf.Tensor:
    image = tf.expand_dims(image, axis=0)
    image = _augmentation_pipeline(image, training=True)
    return tf.squeeze(image, axis=0)


def _decode_and_resize(filepath: tf.Tensor, target_size) -> tf.Tensor:
    raw = tf.io.read_file(filepath)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    return tf.image.resize(image, target_size)


def build_dataset(filepaths, labels, architecture: str, target_size, batch_size: int,
                   shuffle: bool = False, augment_fn: Optional[Callable] = None) -> tf.data.Dataset:
    preprocess_fn = PREPROCESS_FUNCTIONS[architecture]
    ds = tf.data.Dataset.from_tensor_slices((list(filepaths), list(labels)))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=RANDOM_SEED, reshuffle_each_iteration=True)

    def _load(fp, lbl):
        return _decode_and_resize(fp, target_size), lbl
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)

    if augment_fn is not None:
        ds = ds.map(lambda img, lbl: (augment_fn(img), lbl), num_parallel_calls=tf.data.AUTOTUNE)

    def _preprocess(img, lbl):
        return preprocess_fn(img), lbl
    ds = ds.map(_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def load_index(dataset_name: str) -> pd.DataFrame:
    index_path = PROCESSED_ROOT / dataset_name / "index.csv"
    if not index_path.exists():
        raise FileNotFoundError(f"'{index_path}' not found - run CELL 6 first.")
    return pd.read_csv(index_path)


def get_datasets(dataset_name: str, architecture: str):
    """Returns (train_ds, val_ds, test_ds, label_encoder, num_classes)."""
    df = load_index(dataset_name)
    label_encoder = LabelEncoder()
    label_encoder.fit(df.loc[df["split"] == "train", "label"])
    df = df[df["label"].isin(label_encoder.classes_)].copy()
    df["label_idx"] = label_encoder.transform(df["label"])

    target_size = TARGET_SIZE[architecture]

    def subset(split):
        sub = df[df["split"] == split]
        return sub["filepath"].tolist(), sub["label_idx"].tolist()

    train_paths, train_labels = subset("train")
    val_paths, val_labels = subset("val")
    test_paths, test_labels = subset("test")

    train_ds = build_dataset(train_paths, train_labels, architecture, target_size, BATCH_SIZE,
                              shuffle=True, augment_fn=augment_image)
    val_ds = build_dataset(val_paths, val_labels, architecture, target_size, BATCH_SIZE, shuffle=False)
    test_ds = build_dataset(test_paths, test_labels, architecture, target_size, BATCH_SIZE, shuffle=False)

    return train_ds, val_ds, test_ds, label_encoder, len(label_encoder.classes_)


def get_class_weight_dict(dataset_name: str) -> Optional[Dict[int, float]]:
    path = PROCESSED_ROOT / dataset_name / "class_weights.json"
    if not path.exists():
        return None
    with open(path) as f:
        raw = json.load(f)
    return {int(k): float(v) for k, v in raw.items()}

In [ ]:
BASE_CONSTRUCTORS = {
    "vgg16": VGG16,
    "resnet50": ResNet50,
    "mobilenet": MobileNet,
    "densenet121": DenseNet121,
}

WEIGHTS_PATHS = {
    "vgg16": "/kaggle/input/datasets/um/existing-weights/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5",
    "resnet50": "/kaggle/input/datasets/um/existing-weights/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5",
    "mobilenet": "/kaggle/input/datasets/um/existing-weights/mobilenet_1_0_224_tf_no_top.h5",
    "densenet121": "/kaggle/input/datasets/um/existing-weights/densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5",
}


def build_transfer_model(
    architecture: str,
    input_shape,
    num_classes: int,
    dense_units: int = 256,
    dropout_rate: float = 0.30,
    learning_rate: float = 1e-3,
):
    """
    Builds a transfer learning model using locally stored ImageNet weights.
    """

    base_constructor = BASE_CONSTRUCTORS[architecture]

    # Build architecture without downloading weights
    base_model = base_constructor(
        include_top=False,
        weights=None,
        input_shape=input_shape,
    )

    # Load pretrained ImageNet weights from local .h5 file
    base_model.load_weights(WEIGHTS_PATHS[architecture])
    print(f"Loaded ImageNet weights for {architecture}")

    # Freeze backbone
    base_model.trainable = False

    # Classification head
    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(
        inputs=base_model.input,
        outputs=outputs,
        name=f"{architecture}_transfer"
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model, base_model

def unfreeze_top_layers(layers_list, fraction: float = 0.30):
    """
    Unfreezes the top `fraction` of `layers_list`, keeping earlier layers
    frozen and BatchNorm layers always frozen (destabilises fine-tuning on
    small medical datasets otherwise).

    Accepts either:
      - base_model.layers (fresh build, phase1 -> phase2_part1 transition), or
      - model.layers[:-4] (after reloading a saved model - the flattened
        graph construction in build_transfer_model means the backbone's
        layers are the full model's layers minus the 4 head layers appended
        there: GlobalAveragePooling2D, Dense, Dropout, Dense).
    """
    n_layers = len(layers_list)
    n_frozen = int(n_layers * (1 - fraction))
    for layer in layers_list[:n_frozen]:
        layer.trainable = False
    for layer in layers_list[n_frozen:]:
        layer.trainable = not isinstance(layer, tf.keras.layers.BatchNormalization)
    return layers_list


def count_trainable_params(model) -> int:
    return int(sum(tf.size(w).numpy() for w in model.trainable_weights))


def count_total_params(model) -> int:
    return int(model.count_params())


In [ ]:
def _save_fig(fig, save_path: Path):
    save_path = Path(save_path); save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, bbox_inches="tight"); plt.close(fig)
    print(f"[figure] saved -> {save_path}")


def _write_table(df: pd.DataFrame, save_path: Path, index: bool = False):
    save_path = Path(save_path); save_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(save_path.with_suffix(".csv"), index=index)
    with open(save_path.with_suffix(".md"), "w") as f:
        f.write(df.to_markdown(index=index))
    print(f"[table] saved -> {save_path.with_suffix('.csv')} and {save_path.with_suffix('.md')}")


def plot_class_distribution(df, dataset_name, save_path, split="train"):
    counts = df.loc[df["split"] == split, "label"].value_counts().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(counts.index.astype(str), counts.values, color="#4C72B0")
    ax.set_title(f"{dataset_name}: Class Distribution ({split} split)")
    ax.set_xlabel("Class"); ax.set_ylabel("Number of images")
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels(): label.set_ha("right")
    fig.tight_layout(); _save_fig(fig, save_path); return fig


def plot_sample_images(df, dataset_name, save_path, n_per_class=1, max_classes=8):
    classes = df["label"].unique()[:max_classes]
    n_cols = len(classes)
    fig, axes = plt.subplots(n_per_class, n_cols, figsize=(2.0 * n_cols, 2.2 * n_per_class))
    axes = np.atleast_2d(axes)
    for col, cls in enumerate(classes):
        rows = df[df["label"] == cls].sample(min(n_per_class, len(df[df["label"] == cls])), random_state=42)
        for row_idx, (_, row) in enumerate(rows.iterrows()):
            ax = axes[row_idx, col]
            try:
                ax.imshow(Image.open(row["filepath"]).convert("RGB"))
            except Exception as exc:
                ax.text(0.5, 0.5, "unreadable", ha="center", va="center")
                print(f"[warning] could not load '{row['filepath']}': {exc}")
            ax.set_xticks([]); ax.set_yticks([])
            if row_idx == 0: ax.set_title(str(cls), fontsize=9)
    fig.suptitle(f"{dataset_name}: Example Images by Class", y=1.02)
    fig.tight_layout(); _save_fig(fig, save_path); return fig


def plot_model_complexity(params_by_architecture, save_path):
    archs = list(params_by_architecture.keys())
    totals = [params_by_architecture[a]["total"] / 1e6 for a in archs]
    trainables = [params_by_architecture[a]["trainable"] / 1e6 for a in archs]
    x = np.arange(len(archs)); width = 0.35
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(x - width/2, totals, width, label="Total parameters", color="#4C72B0")
    ax.bar(x + width/2, trainables, width, label="Trainable (phase 2)", color="#DD8452")
    ax.set_xticks(x); ax.set_xticklabels([a.upper() if a != "densenet121" else "DenseNet121" for a in archs])
    ax.set_ylabel("Parameters (millions)"); ax.set_title("Model Complexity by Architecture")
    ax.legend(); fig.tight_layout(); _save_fig(fig, save_path); return fig


def plot_training_curves(history, run_name, save_path):
    epochs = range(1, len(history["loss"]) + 1)
    boundary = history.get("phase_boundary")
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    axes[0].plot(epochs, history["accuracy"], label="Train")
    axes[0].plot(epochs, history["val_accuracy"], label="Validation")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy"); axes[0].legend()
    axes[1].plot(epochs, history["loss"], label="Train")
    axes[1].plot(epochs, history["val_loss"], label="Validation")
    axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss"); axes[1].legend()
    if boundary:
        for ax in axes:
            ax.axvline(boundary + 0.5, color="grey", linestyle="--", linewidth=1)
            ax.text(boundary + 0.6, ax.get_ylim()[1]*0.95, "fine-tuning\nstarts", fontsize=7, va="top", color="grey")
    fig.suptitle(f"Training Curves: {run_name}"); fig.tight_layout(); _save_fig(fig, save_path); return fig


def plot_confusion_matrix(cm, class_names, run_name, save_path):
    fig, ax = plt.subplots(figsize=(0.6*len(class_names)+2, 0.6*len(class_names)+2))
    im = ax.imshow(cm, cmap="Blues"); fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted label"); ax.set_ylabel("True label"); ax.set_title(f"Confusion Matrix: {run_name}")
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black", fontsize=8)
    fig.tight_layout(); _save_fig(fig, save_path); return fig


def plot_roc_curves(y_true, y_pred_probs, class_names, run_name, save_path):
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=range(n_classes)) if n_classes > 2 else None
    fig, ax = plt.subplots(figsize=(5, 4.5))
    if n_classes == 2:
        fpr, tpr, _ = roc_curve(y_true, y_pred_probs[:, 1])
        ax.plot(fpr, tpr, label=f"ROC (AUC = {auc(fpr, tpr):.3f})")
    else:
        all_fpr = np.linspace(0, 1, 200); mean_tpr = np.zeros_like(all_fpr)
        for i, cls in enumerate(class_names):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
            mean_tpr += np.interp(all_fpr, fpr, tpr)
            ax.plot(fpr, tpr, alpha=0.35, linewidth=1, label=f"{cls} (AUC={auc(fpr, tpr):.2f})")
        mean_tpr /= n_classes
        ax.plot(all_fpr, mean_tpr, color="black", linewidth=2, label=f"Macro-average (AUC={auc(all_fpr, mean_tpr):.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve: {run_name}"); ax.legend(fontsize=7, loc="lower right")
    fig.tight_layout(); _save_fig(fig, save_path); return fig


def plot_gradcam_grid(originals, overlays, titles, save_path):
    n = len(originals)
    fig, axes = plt.subplots(2, n, figsize=(2.2*n, 4.6)); axes = np.atleast_2d(axes)
    for i in range(n):
        axes[0, i].imshow(originals[i]); axes[0, i].set_xticks([]); axes[0, i].set_yticks([])
        axes[0, i].set_title(titles[i], fontsize=8)
        axes[1, i].imshow(overlays[i]); axes[1, i].set_xticks([]); axes[1, i].set_yticks([])
    axes[0, 0].set_ylabel("Original", fontsize=9); axes[1, 0].set_ylabel("Grad-CAM", fontsize=9)
    fig.tight_layout(); _save_fig(fig, save_path); return fig


def plot_architecture_comparison(results_df, metric, save_path):
    datasets_ = results_df["dataset"].unique(); architectures_ = results_df["architecture"].unique()
    x = np.arange(len(datasets_)); width = 0.8 / len(architectures_)
    fig, ax = plt.subplots(figsize=(7, 4))
    for i, arch in enumerate(architectures_):
        values = [results_df[(results_df["dataset"] == d) & (results_df["architecture"] == arch)][metric].values
                  for d in datasets_]
        values = [v[0] if len(v) else np.nan for v in values]
        ax.bar(x + i*width - 0.4 + width/2, values, width, label=arch)
    ax.set_xticks(x); ax.set_xticklabels(datasets_, rotation=15, ha="right")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(f"Architecture Comparison: {metric.replace('_', ' ').title()}")
    ax.legend(fontsize=8); fig.tight_layout(); _save_fig(fig, save_path); return fig


def dataset_summary_table(df, dataset_name, save_path):
    summary = df.groupby(["label", "split"]).size().unstack(fill_value=0)
    for col in ("train", "val", "test"):
        if col not in summary.columns: summary[col] = 0
    summary = summary[["train", "val", "test"]]
    summary["total"] = summary.sum(axis=1)
    summary = summary.reset_index().rename(columns={"label": "Class"})
    summary.insert(0, "Dataset", dataset_name)
    _write_table(summary, save_path); return summary


def hyperparameters_table(save_path):
    rows = [
        ("Input size (all architectures)", "224 x 224 x 3"),
        ("Batch size", BATCH_SIZE),
        ("Train / val / test split", "70% / 15% / 15% (stratified)"),
        ("Phase 1 epochs (head only, base frozen)", FREEZE_BASE_EPOCHS),
        ("Phase 2 epochs (fine-tuning)", FINE_TUNE_EPOCHS),
        ("Phase 2 unfrozen fraction of base", f"{FINE_TUNE_UNFREEZE_FRACTION:.0%}"),
        ("Phase 1 learning rate", BASE_LEARNING_RATE),
        ("Phase 2 learning rate", FINE_TUNE_LEARNING_RATE),
        ("Optimizer", "Adam"),
        ("Loss function", "Sparse categorical cross-entropy"),
        ("Dropout rate", DROPOUT_RATE),
        ("Dense head units", DENSE_UNITS),
        ("Early stopping patience", f"{EARLY_STOPPING_PATIENCE} epochs (val_loss)"),
        ("LR reduction", f"factor {REDUCE_LR_FACTOR}, patience {REDUCE_LR_PATIENCE} epochs (val_loss)"),
        ("Class imbalance handling", "Class weighting (balanced) / SMOTE (BreakHis only)"),
        ("Augmentation (train split only)", "Flip, rotation (10%), zoom (10%), contrast jitter (10%)"),
        ("Random seed", RANDOM_SEED),
    ]
    df = pd.DataFrame(rows, columns=["Setting", "Value"])
    _write_table(df, save_path); return df


def model_complexity_table(params_by_architecture, save_path):
    rows = [{"Architecture": arch,
             "Total parameters": f"{c['total']:,}",
             "Trainable (phase 2)": f"{c['trainable']:,}",
             "Frozen (phase 2)": f"{c['total'] - c['trainable']:,}"}
            for arch, c in params_by_architecture.items()]
    df = pd.DataFrame(rows); _write_table(df, save_path); return df


def results_table(results, save_path):
    df = pd.DataFrame(results)[["dataset", "architecture", "accuracy", "precision_macro",
                                  "recall_macro", "f1_macro", "auc_roc"]].copy()
    for col in ("accuracy", "precision_macro", "recall_macro", "f1_macro", "auc_roc"):
        df[col] = (df[col] * 100).round(2)
    df = df.rename(columns={"dataset": "Dataset", "architecture": "Architecture",
                              "accuracy": "Accuracy (%)", "precision_macro": "Precision (%)",
                              "recall_macro": "Recall (%)", "f1_macro": "F1-score (%)",
                              "auc_roc": "AUC-ROC (%)"})
    _write_table(df, save_path); return df

In [ ]:
def run_methodology_reporting():
    print("\n=== Generating Methodology-section figures/tables ===")
    loaders = {
        "BreakHis": lambda: load_index("BreakHis"),
        "NCT-CRC-HE-100K": lambda: load_index("NCT-CRC-HE-100K"),
        "ISIC_2019": lambda: load_index("ISIC_2019"),
    }
    for dataset_name, loader in loaders.items():
        df = loader()
        plot_class_distribution(df, dataset_name, FIGURES_METHODOLOGY_DIR / f"{dataset_name}_class_distribution.png")
        plot_sample_images(df, dataset_name, FIGURES_METHODOLOGY_DIR / f"{dataset_name}_sample_images.png")
        dataset_summary_table(df, dataset_name, TABLES_METHODOLOGY_DIR / f"{dataset_name}_dataset_summary")

    hyperparameters_table(TABLES_METHODOLOGY_DIR / "hyperparameters")

    params_by_arch = {}
    for arch in ARCHITECTURES:
        input_shape = TARGET_SIZE[arch] + (3,)
        model, base_model = build_transfer_model(arch, input_shape=input_shape, num_classes=2,
                                                   dense_units=DENSE_UNITS, dropout_rate=DROPOUT_RATE)
        total = count_total_params(model)
        unfreeze_top_layers(base_model.layers, fraction=FINE_TUNE_UNFREEZE_FRACTION)  # <- only change on this line
        trainable = count_trainable_params(model)
        params_by_arch[arch] = {"total": total, "trainable": trainable}
        del model, base_model
        tf.keras.backend.clear_session()

    plot_model_complexity(params_by_arch, FIGURES_METHODOLOGY_DIR / "model_complexity.png")
    model_complexity_table(params_by_arch, TABLES_METHODOLOGY_DIR / "model_complexity")

run_methodology_reporting()

In [ ]:
def build_callbacks(run_name: str, csv_append: bool = False, csv_log_name: str = None):
    """
    run_name is the checkpoint-file base name for THIS STAGE
    (e.g. "BreakHis_vgg16_phase1" or "BreakHis_vgg16_phase2_part2").

    csv_log_name lets multiple stages share one CSVLogger file - all three
    phase2 parts log into the same "{run}_phase2_history.csv" (part1 starts
    it fresh, parts 2 and 3 append), so evaluate() can later reconstruct one
    continuous 15-epoch history from just two CSV files, without needing any
    other persisted state.
    """
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    LOGS_DIR.mkdir(parents=True, exist_ok=True)

    # Best model based on validation loss
    checkpoint_path = MODELS_DIR / f"{run_name}_best.keras"

    # Latest checkpoint (updated every epoch)
    latest_checkpoint_path = MODELS_DIR / f"{run_name}_latest.keras"

    log_path = LOGS_DIR / (csv_log_name or f"{run_name}_history.csv")

    return [
        EarlyStopping(
            monitor="val_loss",
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=REDUCE_LR_FACTOR,
            patience=REDUCE_LR_PATIENCE,
            min_lr=1e-7
        ),

        # Save best model only
        ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        # Save latest model after every epoch
        ModelCheckpoint(
            filepath=str(latest_checkpoint_path),
            monitor="val_loss",
            save_best_only=False,
            save_weights_only=False,
            save_freq="epoch",
            verbose=1
        ),

        CSVLogger(
            str(log_path),
            append=csv_append
        ),
    ]

def train_model(dataset_name: str, architecture: str, train_ds, val_ds, num_classes: int,
                 class_weight: Optional[dict] = None):
    run_name = f"{dataset_name}_{architecture}"
    input_shape = TARGET_SIZE[architecture] + (3,)

    model, base_model = build_transfer_model(architecture, input_shape=input_shape, num_classes=num_classes,
                                               dense_units=DENSE_UNITS, dropout_rate=DROPOUT_RATE,
                                               learning_rate=BASE_LEARNING_RATE)

    print(f"\n=== [{run_name}] Phase 1: training classifier head (base frozen) ===")
    history_1 = model.fit(train_ds, validation_data=val_ds, epochs=FREEZE_BASE_EPOCHS,
                           class_weight=class_weight, callbacks=build_callbacks(f"{run_name}_phase1"))

    print(f"\n=== [{run_name}] Phase 2: fine-tuning top {FINE_TUNE_UNFREEZE_FRACTION:.0%} of base layers ===")
    unfreeze_top_layers(base_model, fraction=FINE_TUNE_UNFREEZE_FRACTION)
    model.compile(optimizer=optimizers.Adam(learning_rate=FINE_TUNE_LEARNING_RATE),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    history_2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
                           class_weight=class_weight, callbacks=build_callbacks(f"{run_name}_phase2"))

    combined_history = {key: list(history_1.history[key]) + list(history_2.history[key])
                         for key in history_1.history}
    combined_history["phase_boundary"] = len(history_1.history["loss"])

    final_path = MODELS_DIR / f"{run_name}_final.keras"
    model.save(final_path)
    print(f"[{run_name}] Saved final model -> {final_path}")

    return model, combined_history


def reconstruct_combined_history(run_name: str) -> dict | None:
    """
    Reconstructs training history from the available CSVLogger files.

    Cases handled:
    1. Both Phase 1 and Phase 2 histories exist -> combine them.
    2. Only Phase 1 exists -> return Phase 1 history.
    3. Only Phase 2 exists -> return Phase 2 history.
    4. Neither exists -> return None (training curve will be skipped).
    """

    phase1_csv = LOGS_DIR / f"{run_name}_phase1_history.csv"
    phase2_csv = LOGS_DIR / f"{run_name}_phase2_history.csv"

    phase1_exists = phase1_csv.exists()
    phase2_exists = phase2_csv.exists()

    if not phase1_exists and not phase2_exists:
        print(f"[WARNING] No training history found for {run_name}.")
        return None

    if phase1_exists:
        df1 = pd.read_csv(phase1_csv)

    if phase2_exists:
        df2 = pd.read_csv(phase2_csv)

    # Both histories available
    if phase1_exists and phase2_exists:
        return {
            "loss": df1["loss"].tolist() + df2["loss"].tolist(),
            "val_loss": df1["val_loss"].tolist() + df2["val_loss"].tolist(),
            "accuracy": df1["accuracy"].tolist() + df2["accuracy"].tolist(),
            "val_accuracy": df1["val_accuracy"].tolist() + df2["val_accuracy"].tolist(),
            "phase_boundary": len(df1),
        }

    # Only Phase 1 available
    elif phase1_exists:
        print(f"[WARNING] Phase 2 history missing for {run_name}. Plotting Phase 1 only.")

        return {
            "loss": df1["loss"].tolist(),
            "val_loss": df1["val_loss"].tolist(),
            "accuracy": df1["accuracy"].tolist(),
            "val_accuracy": df1["val_accuracy"].tolist(),
            "phase_boundary": len(df1),
        }

    # Only Phase 2 available
    else:
        print(f"[WARNING] Phase 1 history missing for {run_name}. Plotting Phase 2 only.")

        return {
            "loss": df2["loss"].tolist(),
            "val_loss": df2["val_loss"].tolist(),
            "accuracy": df2["accuracy"].tolist(),
            "val_accuracy": df2["val_accuracy"].tolist(),
            "phase_boundary": 0,
        }

STAGE_ORDER = ["phase1", "phase2_part1", "phase2_part2", "phase2_part3", "evaluate"]

In [ ]:
def run_stage(dataset_name: str, architecture: str, stage: str, class_weight: dict = None):
    """
    Runs exactly one stage for one (dataset, architecture) combination.
    Self-contained: everything it needs (index.csv, class_weights.json, and
    for resuming stages the previous stage's saved .keras file) is loaded
    from disk, so this works correctly even when called in a fresh Kaggle
    kernel that has no memory of any earlier session.
    """
    if stage not in STAGE_ORDER:
        raise ValueError(f"Unknown stage '{stage}'. Choose from {STAGE_ORDER}.")

    run_name = f"{dataset_name}_{architecture}"
    print(f"\n{'='*70}\nRUN: {run_name}  |  STAGE: {stage}\n{'='*70}")

    train_ds, val_ds, test_ds, label_encoder, num_classes = get_datasets(dataset_name, architecture)
    class_names = list(label_encoder.classes_)

    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    with open(MODELS_DIR / f"{run_name}_classes.json", "w") as f:
        json.dump(class_names, f, indent=2)

    if stage == "evaluate":
        return _run_evaluate_stage(run_name, dataset_name, architecture, test_ds, class_names, num_classes)

    _run_training_stage(run_name, architecture, stage, train_ds, val_ds, num_classes, class_weight)
    return None


def _run_training_stage(run_name: str, architecture: str, stage: str,
                         train_ds, val_ds, num_classes: int, class_weight: dict = None):
    input_shape = TARGET_SIZE[architecture] + (3,)
    from pathlib import Path

    if stage == "phase1":

        latest_checkpoint = MODELS_DIR / f"{run_name}_phase1_latest.keras"

        if latest_checkpoint.exists():
            print(f"Resuming Phase 1 from checkpoint: {latest_checkpoint}")

            # Load latest checkpoint (includes optimizer state)
            model = tf.keras.models.load_model(latest_checkpoint)

            # -------------------------------------------------------
            # MANUALLY SET THIS TO THE NUMBER OF COMPLETED EPOCHS
            # Example:
            # completed_epochs = 2
            # means Epochs 1 & 2 are complete, so training resumes
            # from Epoch 3.
            # -------------------------------------------------------
            completed_epochs = 0

            initial_epoch = completed_epochs
            epochs = FREEZE_BASE_EPOCHS

            callbacks = build_callbacks(
                f"{run_name}_phase1",
                csv_append=True
            )

        else:
            print("No Phase 1 checkpoint found. Starting from scratch.")

            model, _base_model = build_transfer_model(
                architecture,
                input_shape=input_shape,
                num_classes=num_classes,
                dense_units=DENSE_UNITS,
                dropout_rate=DROPOUT_RATE,
                learning_rate=BASE_LEARNING_RATE,
            )

            initial_epoch = 0
            epochs = FREEZE_BASE_EPOCHS

            callbacks = build_callbacks(
                f"{run_name}_phase1",
                csv_append=False
            )

    else:
        # phase2_part1 / phase2_part2 / phase2_part3
        phase2_config = {
            "phase2_part1": dict(
                load_from=f"{run_name}_phase1_final.keras",
                initial_epoch=0,
                epochs=5,
                fresh_compile=True,
            ),
            "phase2_part2": dict(
                load_from=Path("/kaggle/input/datasets/um/p2p2-final-resnet-isic/ISIC_2019_resnet50_phase2_part1_final.keras"),
                initial_epoch=5,
                epochs=10,
                fresh_compile=False,
            ),
            "phase2_part3": dict(
                load_from=f"{run_name}_phase2_part2_final.keras",
                initial_epoch=10,
                epochs=FINE_TUNE_EPOCHS,
                fresh_compile=False,
            ),
        }[stage]
        PHASE1_MODEL_DIR = Path("/kaggle/input/datasets/um/isic-phase-1-final")
        PHASE2_CHECKPOINT_DIR = Path("/kaggle/working/outputs/models")
        
        WORKING_MODELS_DIR = MODELS_DIR
        INPUT_MODELS_DIR = Path("/kaggle/working/outputs/models")

        # -------------------------------------------------------
        # FIRST: look for the stage's latest checkpoint
        # -------------------------------------------------------
        latest_checkpoint = WORKING_MODELS_DIR / f"{run_name}_{stage}_latest.keras"

        if not latest_checkpoint.exists():
            latest_checkpoint = PHASE2_CHECKPOINT_DIR / f"{run_name}_{stage}_latest.keras"

        if latest_checkpoint.exists():

            print(f"Resuming {stage} from checkpoint: {latest_checkpoint}")

            # Load checkpoint including optimizer state
            model = tf.keras.models.load_model(latest_checkpoint)

            # -------------------------------------------------------
            # MANUALLY SET THIS TO THE NUMBER OF COMPLETED EPOCHS
            #
            # Examples:
            #
            # phase2_part1:
            # completed_epochs = 3
            #
            # phase2_part2:
            # completed_epochs = 7
            #
            # phase2_part3:
            # completed_epochs = 12
            #
            # Resume always starts from completed_epochs.
            # -------------------------------------------------------
            completed_epochs = 12

            initial_epoch = completed_epochs
            epochs = phase2_config["epochs"]

            callbacks = build_callbacks(
                f"{run_name}_{stage}",
                csv_append=True,
                csv_log_name=f"{run_name}_phase2_history.csv",
            )

        else:

            print(f"No latest checkpoint found for {stage}. Starting from stage beginning.")

            # Try working directory first
            load_path = WORKING_MODELS_DIR / phase2_config["load_from"]

            if not load_path.exists():
                load_path = PHASE1_MODEL_DIR / phase2_config["load_from"]

            if not load_path.exists():
                raise FileNotFoundError(
                    f"Model '{phase2_config['load_from']}' not found in either:\n"
                    f"  {WORKING_MODELS_DIR}\n"
                    f"  {INPUT_MODELS_DIR}"
                )

            if phase2_config["fresh_compile"]:
                # First entry into phase 2
                model = tf.keras.models.load_model(load_path, compile=False)

                backbone_layers = model.layers[:-4]

                unfreeze_top_layers(
                    backbone_layers,
                    fraction=FINE_TUNE_UNFREEZE_FRACTION,
                )

                model.compile(
                    optimizer=optimizers.Adam(
                        learning_rate=FINE_TUNE_LEARNING_RATE
                    ),
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"],
                )

            else:
                # Continue fine-tuning with optimizer state intact
                model = tf.keras.models.load_model(load_path)

            initial_epoch = phase2_config["initial_epoch"]
            epochs = phase2_config["epochs"]

            callbacks = build_callbacks(
                f"{run_name}_{stage}",
                csv_append=(stage != "phase2_part1"),
                csv_log_name=f"{run_name}_phase2_history.csv",
            )

    model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=initial_epoch,
        epochs=epochs,
        class_weight=class_weight,
        callbacks=callbacks,
    )

    stage_final_path = MODELS_DIR / f"{run_name}_{stage}_final.keras"
    model.save(stage_final_path)

    print(f"[{run_name}] Stage '{stage}' complete -> {stage_final_path}")

    if stage == "phase2_part3":
        final_path = MODELS_DIR / f"{run_name}_final.keras"
        model.save(final_path)
        print(f"[{run_name}] Training complete -> {final_path}")

    tf.keras.backend.clear_session()

In [ ]:
def evaluate_model(model, test_ds, num_classes: int) -> Dict:
    y_true, y_pred_probs = [], []
    for images, labels in test_ds:
        y_pred_probs.append(model.predict(images, verbose=0))
        y_true.append(labels.numpy())

    y_true = np.concatenate(y_true)
    y_pred_probs = np.concatenate(y_pred_probs)
    y_pred = np.argmax(y_pred_probs, axis=1)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    try:
        if num_classes == 2:
            metrics["auc_roc"] = roc_auc_score(y_true, y_pred_probs[:, 1])
        else:
            metrics["auc_roc"] = roc_auc_score(y_true, y_pred_probs, multi_class="ovr", average="macro")
    except ValueError as exc:
        print(f"[warning] AUC-ROC could not be computed: {exc}")
        metrics["auc_roc"] = float("nan")

    metrics["confusion_matrix"] = confusion_matrix(y_true, y_pred)
    metrics["y_true"] = y_true
    metrics["y_pred"] = y_pred
    metrics["y_pred_probs"] = y_pred_probs
    return metrics

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name: str, pred_index: int = None):
    grad_model = tf.keras.models.Model(inputs=model.inputs,
                                        outputs=[model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = int(tf.argmax(predictions[0]))
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_index


def overlay_heatmap(image_uint8: np.ndarray, heatmap: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    import matplotlib.cm as cm
    heatmap_resized = tf.image.resize(
        heatmap[..., tf.newaxis], (image_uint8.shape[0], image_uint8.shape[1])).numpy().squeeze()
    jet = cm.get_cmap("jet")
    jet_colors = (jet(heatmap_resized)[:, :, :3] * 255).astype(np.uint8)
    return (jet_colors * alpha + image_uint8 * (1 - alpha)).astype(np.uint8)


def run_gradcam_examples(model, test_ds, architecture, class_names, run_name):
    last_conv_layer = GRADCAM_LAST_CONV_LAYER[architecture]
    originals, overlays, titles = [], [], []
    for images, labels in test_ds.take(1):
        images = images.numpy(); labels = labels.numpy()
        for i in range(min(GRADCAM_NUM_EXAMPLES, len(images))):
            heatmap, pred_idx = make_gradcam_heatmap(images[i:i+1], model, last_conv_layer)
            display_img = images[i] - images[i].min()
            display_img = (display_img / (display_img.max() + 1e-8) * 255).astype(np.uint8)
            overlays.append(overlay_heatmap(display_img, heatmap))
            originals.append(display_img)
            titles.append(f"true={class_names[labels[i]]}\npred={class_names[pred_idx]}")
        break
    if originals:
        plot_gradcam_grid(originals, overlays, titles, FIGURES_RESULTS_DIR / f"{run_name}_gradcam.png")


In [ ]:
def run_training(dataset_names: list, architectures: list, stage: str) -> list:
    """
    stage : one of "phase1", "phase2_part1", "phase2_part2", "phase2_part3", "evaluate".

    Example (run each of these in a separate Kaggle session):
        run_training(dataset_names=["BreakHis"], architectures=["vgg16"], stage="phase1")
        run_training(dataset_names=["BreakHis"], architectures=["vgg16"], stage="phase2_part1")
        run_training(dataset_names=["BreakHis"], architectures=["vgg16"], stage="phase2_part2")
        run_training(dataset_names=["BreakHis"], architectures=["vgg16"], stage="phase2_part3")
        results = run_training(dataset_names=["BreakHis"], architectures=["vgg16"], stage="evaluate")

    Training stages return []; "evaluate" returns the same list-of-dicts
    shape run_training() always returned, ready for run_results_reporting().
    """
    if stage not in STAGE_ORDER:
        raise ValueError(f"Unknown stage '{stage}'. Choose from {STAGE_ORDER}.")

    results = []
    for dataset_name in dataset_names:
        class_weight = get_class_weight_dict(dataset_name) if stage != "evaluate" else None
        for architecture in architectures:
            outcome = run_stage(dataset_name, architecture, stage, class_weight=class_weight)
            if stage == "evaluate":
                results.append(outcome)

    return results

from pathlib import Path

def find_best_model_path(run_name: str) -> Path:
    candidates = [
        MODELS_DIR / f"{run_name}_final.keras",
        MODELS_DIR / f"{run_name}_phase2_part3_final.keras",
        MODELS_DIR / f"{run_name}_phase2_part2_final.keras",
        MODELS_DIR / f"{run_name}_phase2_part1_final.keras",
        MODELS_DIR / f"{run_name}_phase1_final.keras",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        f"No saved model found for {run_name}. Checked:\n" +
        "\n".join(str(p) for p in candidates)
    )

def _run_evaluate_stage(run_name: str, dataset_name: str, architecture: str,
                        test_ds, class_names: list, num_classes: int) -> dict:
    final_path = find_best_model_path(run_name)
    model = tf.keras.models.load_model(final_path)

    history = reconstruct_combined_history(run_name)
    if history is not None:
        plot_training_curves(history, run_name, FIGURES_RESULTS_DIR / f"{run_name}_training_curves.png")
    else:
        print("Skipping training curve because no history was found.")
    metrics = evaluate_model(model, test_ds, num_classes)
    plot_confusion_matrix(metrics["confusion_matrix"], class_names, run_name,
                          FIGURES_RESULTS_DIR / f"{run_name}_confusion_matrix.png")
    plot_roc_curves(metrics["y_true"], metrics["y_pred_probs"], class_names, run_name,
                    FIGURES_RESULTS_DIR / f"{run_name}_roc_curve.png")
    run_gradcam_examples(model, test_ds, architecture, class_names, run_name)

    result = {
        "dataset": dataset_name,
        "architecture": architecture,
        "accuracy": metrics["accuracy"],
        "precision_macro": metrics["precision_macro"],
        "recall_macro": metrics["recall_macro"],
        "f1_macro": metrics["f1_macro"],
        "auc_roc": metrics["auc_roc"],
    }
    tf.keras.backend.clear_session()
    return result

In [ ]:
# STAGE: evaluate — loads each {dataset}_{arch}_final.keras, reconstructs the
# full 15-epoch training-curve history from the persisted CSV logs, and runs
# evaluation: confusion matrix, ROC curve, Grad-CAM, and metrics. No training
# happens in this stage.
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["vgg16"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_vgg16_results"
)

# Optional: display in notebook
print(pd.DataFrame(all_results))

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="phase1",
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="phase2_part3",
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_resnet50_results"
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="phase1",
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="phase2_part3",
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_mobilenet_results"
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_densenet121_results"
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_densenet121_results"
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_densenet121_results"
)

In [ ]:
all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="phase1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["resnet50"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_resnet50_results"
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="phase1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["mobilenet"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_mobilenet_results"
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["BreakHis"],
    architectures=["densenet121"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "BreakHis_densenet121_results"
)

all_results = run_training(
    dataset_names=["NCT-CRC-HE-100K"],
    architectures=["vgg16"],
    stage="phase1",
)

all_results = run_training(
    dataset_names=["NCT-CRC-HE-100K"],
    architectures=["vgg16"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["NCT-CRC-HE-100K"],
    architectures=["vgg16"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["NCT-CRC-HE-100K"],
    architectures=["vgg16"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["NCT-CRC-HE-100K"],
    architectures=["vgg16"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "NCT-CRC-HE-100K_vgg16_results"
)

In [ ]:
from pathlib import Path
from PIL import Image

src = Path("/kaggle/input/datasets/um/cancer-cnn-classification/NCT-CRC-HE-100K")
dst = Path("/kaggle/working/NCT-CRC-HE-100K-PNG")

for tif_file in src.rglob("*.tif"):
    relative = tif_file.relative_to(src)
    png_file = (dst / relative).with_suffix(".png")

    png_file.parent.mkdir(parents=True, exist_ok=True)

    with Image.open(tif_file) as img:
        img.save(png_file, "PNG")

print("Conversion complete!")

In [ ]:
all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["vgg16"],
    stage="phase1",
)

In [ ]:
all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["vgg16"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["vgg16"],
    stage="phase2_part2",
)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working/output"):
    for f in files:
        if f.endswith(".keras"):
            print(os.path.join(root, f))

In [ ]:
from pathlib import Path

print(list(Path("/kaggle/input/datasets/um/isic-phase-1-final").glob("*")))
print(list(Path("/kaggle/input/datasets/um/phase2-part-1-epcoh").glob("*")))

In [ ]:
all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["vgg16"],
    stage="phase2_part3",
)

In [ ]:
all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["vgg16"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "ISIC_vgg16_results"
)

In [ ]:
all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="phase1",
)

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="phase2_part1",
)

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="evaluate",
)

esults_table(
    all_results,
    TABLES_RESULTS_DIR / "ISIC_vgg16_results"
)

In [ ]:

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="phase2_part2",
)

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "ISIC_vgg16_results"
)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working/output"):
    for f in files:
        if f.endswith(".keras"):
            print(os.path.join(root, f))

In [ ]:
print(MODELS_DIR.exists())
print(list(MODELS_DIR.iterdir()) if MODELS_DIR.exists() else "Directory doesn't exist")

In [ ]:
all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="phase2_part3",
)

all_results = run_training(
    dataset_names=["ISIC_2019"],
    architectures=["resnet50"],
    stage="evaluate",
)

results_table(
    all_results,
    TABLES_RESULTS_DIR / "ISIC_vgg16_results"
)


RUN: ISIC_2019_resnet50  |  STAGE: phase2_part3
Resuming phase2_part3 from checkpoint: /kaggle/working/outputs/models/ISIC_2019_resnet50_phase2_part3_latest.keras
Epoch 13/15
 21/555 ━━━━━━━━━━━━━━━━━━━━ 43:30 5s/step - accuracy: 0.7872 - loss: 0.4424